## Making initial workflows 

`trovolut` works by creating initial workflows which then can be optimized using openevolve. The `filemaker` module contains tools to create these initial workflows.

In [6]:
import trovolut as trv

In [7]:
import cupynumeric as np

# creates an initial state vector
def initial_state_prep(n):
    psi0 = np.zeros(n, dtype=np.complex64)
    psi0[0] = 1.0
    return psi0

# makes a first-order Trotter time evolution
def trotter_time_evol(H, psi0, t, r):
    dt = t / r
    psi = psi0
    for _ in range(r):
        for H_j in H:
            U_j = np.linalg.expm(-1j * H_j * dt)
            psi = U_j @ psi
    psi_final = psi

    return psi_final

# runs the dynamics
def run_dynamics():
    n = 8
    t = 1.0
    r = 10

    hamcoeffs = trv.qm.models.ising_1d(n, h=1.0, J=1.0).terms()
    mats = [term['coef'] * term['matrix'] for term in hamcoeffs]

    psi0 = initial_state_prep(mats[0].shape[0])
    psi_final = trotter_time_evol(mats, psi0, t, r)
    return psi_final

def main():
    st = run_dynamics()
    print("Shape of final state:", st.shape)

init_program = trv.filemaker.Workflow(
    [
        initial_state_prep,
        trv.filemaker.EVO_START,
        trotter_time_evol,
        trv.filemaker.EVO_END,
        run_dynamics,
    ],
    main_func=main,
    out_path="opt/init_program.py",
).build()

print("Wrote", init_program)

Wrote opt/init_program.py


In [8]:
with open(init_program, "r") as f:
    print(f.read())

import cupynumeric as np
import trovolut as trv

def initial_state_prep(n):
    psi0 = np.zeros(n, dtype=np.complex64)
    psi0[0] = 1.0
    return psi0

# EVOLVE-BLOCK-START

def trotter_time_evol(H, psi0, t, r):
    dt = t / r
    psi = psi0
    for _ in range(r):
        for H_j in H:
            U_j = np.linalg.expm(-1j * H_j * dt)
            psi = U_j @ psi
    psi_final = psi

    return psi_final

# EVOLVE-BLOCK-END

def run_dynamics():
    n = 8
    t = 1.0
    r = 10

    hamcoeffs = trv.qm.models.ising_1d(n, h=1.0, J=1.0).terms()
    mats = [term['coef'] * term['matrix'] for term in hamcoeffs]

    psi0 = initial_state_prep(mats[0].shape[0])
    psi_final = trotter_time_evol(mats, psi0, t, r)
    return psi_final

if __name__ == "__main__":
        st = run_dynamics()
        print("Shape of final state:", st.shape)




This file then can be run as a standalone Python script. This is useful as a starting point for code optimization.

In [9]:
import subprocess

result = subprocess.run(
    ["uv", "run", "python", str(init_program)],
    capture_output=True,
    text=True
)

print(result.stdout)
# print(result.stderr)

Shape of final state: (256,)

